# CodiEsp - 1000 Clinical Cases - Knowledge Graph

Data responses were generated from GPT-4o using a prompt template which included the clinical cases: ICD-10 diagnoses codes and the clinical notes.  Each ICD code along with the clinical notes was used to determine the relations type between the ICD-10 titles (Casual, Risk Factors, Associations, or no direct relationship)  

<hr style="height:1rem;">

## 1.0 Imports

In [1]:
import pandas as pd

import json
import os

import networkx as nx
import matplotlib.pyplot as plt

from pyvis.network import Network
import re
from collections import defaultdict

<hr style="height:1rem;">

## 2.0 Function Declarations

In [2]:
def merge_jsons(json_list):
    merged = {"nodes": [], "edges": []}
    
    node_ids = set()
    edge_set = set()
    
    for json_data in json_list:
        for node in json_data.get("nodes", []):
            if node["id"] not in node_ids:
                merged["nodes"].append(node)
                node_ids.add(node["id"])
        
        for edge in json_data.get("edges", []):
            edge_tuple = (edge["source"], edge["target"], edge["relationship"])
            if edge_tuple not in edge_set:
                merged["edges"].append(edge)
                edge_set.add(edge_tuple)

    return merged

In [3]:
def json_to_dataframe(json_data, record_path=None, meta=None):
    """
    Convert JSON data into a pandas DataFrame.

    Parameters:
    - json_data: The input JSON data (dict or list).
    - record_path: (Optional) Path to nested list if the JSON is nested.
    - meta: (Optional) List of fields to pull from parent record in nested JSON.

    Returns:
    - pd.DataFrame: The resulting DataFrame.
    """
    if isinstance(json_data, list):
        return pd.json_normalize(json_data, record_path=record_path, meta=meta)
    elif isinstance(json_data, dict):
        # Try to infer a suitable structure
        if "nodes" in json_data and "edges" in json_data:
            nodes_df = pd.DataFrame(json_data["nodes"])
            edges_df = pd.DataFrame(json_data["edges"])
            return nodes_df, edges_df
        else:
            return pd.json_normalize(json_data)
    else:
        raise ValueError("Input must be a dictionary or list of dictionaries.")

<hr style="height:1rem;">

## 3.0 Read Data into DataFrame

In [4]:
#FILE = 'KNOWLEDGE_GRAPH_RESULTS_1000.xlsx'

#FILE = 'Clincal_Case_A.xlsx'
#FILE = 'Clincal_Case_B.xlsx'
FILE = 'Clincal_Case_C.xlsx'
#FILE = 'Clincal_Case_D.xlsx'
#FILE = 'Clincal_Case_E.xlsx'
#FILE = 'Clincal_Case_F.xlsx'

df = pd.read_excel(FILE)
df.head(5)

,article_id,text_references,icd_codes,text,prompt_template,is_icd_in_template,response
0,S1130-01082008001000008-1,abdominal wall hematoma; DMNID; right anterior...,M79.81 E11.9 R05 T14.8 BW20,A 61-year-old male with a history of IDDM was ...,You are an expert in medical NLP and knowledge...,True,"{\n ""nodes"": [\n {\n ""id"": ""M79.81"",\..."


In [5]:
json_list = []
# Loop through DataFrame
for index, row in df.iterrows():
    json_data = json.loads(row["response"])
    json_list.append(json_data)

data = merge_jsons(json_list)

In [6]:
data

{'nodes': [{'id': 'M79.81',
   'name': 'Nontraumatic hematoma of soft tissue',
   'type': 'Disease'},
  {'id': 'E11.9',
   'name': 'Type 2 diabetes mellitus without complications',
   'type': 'Disease'},
  {'id': 'R05', 'name': 'Cough', 'type': 'Symptom'},
  {'id': 'T14.8',
   'name': 'Other injury of unspecified body region',
   'type': 'Injury'}],
 'edges': [{'source': 'M79.81',
   'target': 'T14.8',
   'relationship': 'Complication of',
   'confidence': 'High',
   'supporting_text': 'A blow of cough followed by a bulging of the entire right anterior rectum, very painful and hard consistency.'},
  {'source': 'E11.9',
   'target': 'M79.81',
   'relationship': 'Associated with',
   'confidence': 'Moderate',
   'supporting_text': 'The patient has a history of IDDM (Insulin-dependent diabetes mellitus), which could contribute to the development of soft tissue hematoma.'},
  {'source': 'R05',
   'target': 'T14.8',
   'relationship': 'Cause of',
   'confidence': 'High',
   'supporting_text

In [7]:
df_example = json_to_dataframe(data)
df_example

(       id                                            name     type
 0  M79.81            Nontraumatic hematoma of soft tissue  Disease
 1   E11.9  Type 2 diabetes mellitus without complications  Disease
 2     R05                                           Cough  Symptom
 3   T14.8         Other injury of unspecified body region   Injury,
    source  target     relationship confidence  \
 0  M79.81   T14.8  Complication of       High   
 1   E11.9  M79.81  Associated with   Moderate   
 2     R05   T14.8         Cause of       High   
 
                                      supporting_text  
 0  A blow of cough followed by a bulging of the e...  
 1  The patient has a history of IDDM (Insulin-dep...  
 2  The patient presented with a blow of cough, wh...  )

<hr style="height:1rem;">

## 5.0 Build Knowledge Graph

### Show a Subgraph based on an ICD-10 Code Entered -- Will show all the edges and nodes

In [ ]:
# Create the full network
net = Network(notebook=True, height="1000px", width="100%", directed=True)

# Add nodes
node_dict = {node["id"]: node for node in data["nodes"]}
for node in data["nodes"]:
    label_name = f'{node["id"]} ({node["name"]})'
    net.add_node(node["id"], label=label_name, title=node["type"], color="lightblue")

# Add edges
edges_list = []
for edge in data["edges"]:
    if edge["source"] in node_dict and edge["target"] in node_dict:
        net.add_edge(
            edge["source"], edge["target"],
            title=f'{edge["relationship"]} ({str(edge["confidence"])})'
        )
        edges_list.append(edge)

# Save full graph
net.show("full_knowledge_graph.html")
print("Full knowledge graph saved as 'full_knowledge_graph.html'.")

# Build node degree info (optional helper)
node_degrees = defaultdict(int)
for edge in edges_list:
    node_degrees[edge["source"]] += 1
    node_degrees[edge["target"]] += 1

# Continuous loop for user input
while True:
    node_input = input("\nEnter node ID(s) (comma-separated) (i.e. M79.81, E11.9, R05, T14.8) to explore (or type 'exit' to quit): ").strip()
    
    if node_input.lower() == "exit":
        print("Exiting program.")
        break

    # Parse and sanitize input
    selected_nodes = [nid.strip() for nid in node_input.split(",") if nid.strip()]
    valid_nodes = [nid for nid in selected_nodes if nid in node_dict]

    if not valid_nodes:
        print("No valid node IDs found. Please try again.")
        continue

    # Create subgraph
    sub_net = Network(notebook=True, height="800px", width="80%", directed=True)

    added_nodes = set()
    added_edges = set()

    subgraph_nodes = []
    subgraph_edges = []

    for node_id in valid_nodes:
        node_data = node_dict[node_id]
        sub_net.add_node(node_id,
                         label=f'{node_id} ({node_data["name"]})',
                         title=node_data["type"],
                         color="red")
        added_nodes.add(node_id)
        subgraph_nodes.append({
            "id": node_id,
            "name": node_data["name"],
            "type": node_data["type"]
        })

        # Add direct neighbors and edges
        for edge in edges_list:
            if edge["source"] == node_id or edge["target"] == node_id:
                source = edge["source"]
                target = edge["target"]
                other_node = target if source == node_id else source

                if other_node not in added_nodes:
                    other_data = node_dict[other_node]
                    sub_net.add_node(other_node,
                                     label=f'{other_node} ({other_data["name"]})',
                                     title=other_data["type"],
                                     color="lightblue")
                    added_nodes.add(other_node)
                    subgraph_nodes.append({
                        "id": other_node,
                        "name": other_data["name"],
                        "type": other_data["type"]
                    })

                edge_id = (source, target)
                if edge_id not in added_edges:
                    sub_net.add_edge(source, target,
                                     title=f'{edge["relationship"]} ({str(edge["confidence"])})')
                    added_edges.add(edge_id)
                    subgraph_edges.append({
                        "source": source,
                        "target": target,
                        "relationship": edge["relationship"],
                        "confidence": edge["confidence"]
                    })

    # Save and show subgraph
    safe_name = re.sub(r'\W+', '_', "_".join(valid_nodes))
    subgraph_filename = f"subgraph_{safe_name}.html"
    sub_net.show(subgraph_filename)
    print(f"\nSubgraph saved as '{subgraph_filename}'. Open it in your browser to view.")

    # Print nodes
    print("\nNodes in subgraph:")
    for node in subgraph_nodes:
        print(f"  - ID: {node['id']}, Name: {node['name']}, Type: {node['type']}")

    # Print edges
    print("\nEdges in subgraph:")
    for edge in subgraph_edges:
        print(f"  - {edge['source']} → {edge['target']} | Relationship: {edge['relationship']} (Confidence: {edge['confidence']})")


    df_nodes = json_to_dataframe(subgraph_nodes)
    df_nodes.to_csv(f"./generated_kg_pd/{safe_name}_nodes.csv", index=False)

    df_edges = json_to_dataframe(subgraph_edges)
    df_edges.to_csv(f"./generated_kg_pd/{safe_name}_edges.csv", index=False)
    
    

full_knowledge_graph.html
Full knowledge graph saved as 'full_knowledge_graph.html'.


In [ ]:
# Create the full network
net = Network(notebook=True, height="1000px", width="100%", directed=True)

# Add nodes
node_dict = {node["id"]: node for node in data["nodes"]}
for node in data["nodes"]:
    label_name = f'{node["id"]} ({node["name"]})'
    net.add_node(node["id"], label=label_name, title=node["type"], color="lightblue")

# Add edges
edges_list = []
for edge in data["edges"]:
    if edge["source"] in node_dict and edge["target"] in node_dict:
        net.add_edge(
            edge["source"], edge["target"],
            title=f'{edge["relationship"]} ({str(edge["confidence"])})'
        )
        edges_list.append(edge)

# Save full graph
net.show("full_knowledge_graph.html")
print("Full knowledge graph saved as 'full_knowledge_graph.html'.")

# Build node degree info (optional helper)
node_degrees = defaultdict(int)
for edge in edges_list:
    node_degrees[edge["source"]] += 1
    node_degrees[edge["target"]] += 1

# Continuous loop for user input
while True:
    node_input = input("\nEnter node ID(s) (comma-separated) to explore (or type 'exit' to quit): ").strip()
    
    if node_input.lower() == "exit":
        print("Exiting program.")
        break

    # Parse and sanitize input
    selected_nodes = [nid.strip() for nid in node_input.split(",") if nid.strip()]
    valid_nodes = [nid for nid in selected_nodes if nid in node_dict]

    if not valid_nodes:
        print("No valid node IDs found. Please try again.")
        continue

    # Create subgraph
    sub_net = Network(notebook=True, height="800px", width="80%", directed=True)

    added_nodes = set()
    added_edges = set()

    for node_id in valid_nodes:
        sub_net.add_node(node_id,
                         label=f'{node_id} ({node_dict[node_id]["name"]})',
                         title=node_dict[node_id]["type"],
                         color="red")
        added_nodes.add(node_id)

        # Add direct neighbors and edges
        for edge in edges_list:
            if edge["source"] == node_id or edge["target"] == node_id:
                source = edge["source"]
                target = edge["target"]
                other_node = target if source == node_id else source

                if other_node not in added_nodes:
                    sub_net.add_node(other_node,
                                     label=f'{other_node} ({node_dict[other_node]["name"]})',
                                     title=node_dict[other_node]["type"],
                                     color="lightblue")
                    added_nodes.add(other_node)

                edge_id = (source, target)
                if edge_id not in added_edges:
                    sub_net.add_edge(source, target,
                                     title=f'{edge["relationship"]} ({str(edge["confidence"])})')
                    added_edges.add(edge_id)

    # Save and show subgraph
    safe_name = re.sub(r'\W+', '_', "_".join(valid_nodes))
    subgraph_filename = f"subgraph_{safe_name}.html"
    sub_net.show(subgraph_filename)
    print(f"Subgraph saved as '{subgraph_filename}'. Open it in your browser to view.")